# A2.1 · Agent identity: user, workload, agent

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A1.18 · The CyberTravels risk register](https://spbreed.github.io/cyber-commons/lessons/A1.18.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE, Keycloak |

## What this lesson is

**What it covers.** Separate the three identities and show a downstream service authorising on the agent while attributing to the human.

**Why a security engineer needs it.** A shared service account answers 'what ran' and destroys 'for whom' — so no later control can be conditioned on the caller. The control it builds is: a distinct identity per workload, carrying the human principal alongside it, asserted on every call.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Three identities are present every time an agent acts: the person who asked, the workload that runs, and the agent instance doing the work. Collapse any two of them and you lose the ability to answer the only question that matters after an incident.

> **At CyberTravels.** Three identities are present whenever CyberTravels books a flight: the traveller who asked, the workload the agent runs as, and which of the four agents is acting. CyberTravels collapses all three into `cybertravels-svc`, which is R11.

## 2 · The framework

```
   who asked        what runs         what acted
   +----------+     +-----------+     +--------------+
   |  human   | --> | workload  | --> | agent        |
   | dana@..  |     | pod/task  |     | instance #7  |
   +----------+     +-----------+     +--------------+
        |                |                   |
      consent         attestation        the actor in the log

   collapse any two and the post-incident question loses its answer
```

**Mitigates: T9 Identity Spoofing · T3 Privilege Compromise · T1 Memory Poisoning.**

Three identities are in play whenever an agent acts, and A1.6 and A1.7 were both
caused by collapsing them into one.

**The user.** The human who asked. Carries the business authority and is the
answer to "on whose behalf".

**The workload.** The process that runs — a container, a function, a pod. It has
its own identity, derived from the platform, not from a secret someone pasted.

**The agent instance.** This particular run, of this particular agent, for this
particular task. It is what you revoke when one agent misbehaves.

The control is to keep all three, and to use them for different things:

- **Authorize on the workload.** What may this agent ever do? That is its
  ceiling, and it does not change per request.
- **Attribute to the user.** Who caused this? That is what the audit trail needs
  and what A1.14 could not answer.
- **Scope memory and state to the instance or the user**, never to the workload
  alone — which is the write that made A1.4 spread across sessions.

The failure mode to watch for is a system that authenticates the agent and then
forgets the human, because it produces logs that are complete and useless.

> **What this control closes.**
>
> Answers **who is calling**, so every later control has a subject. Without it, default-deny has nothing to deny and the audit trail has nobody to name.

## 3 · Separating the three, as a skill

Keeping the three identities apart is a review you will run against every agent CyberTravels ships, not a decision you make once. So it is written down as a procedure: which principal authorises, which one is attributed, which one scopes memory — and the two narrowing rules a delegated token has to satisfy before any of it means anything. This is the file in this repository, embedded verbatim:

In [ ]:
# skills/identity/agent-identity-review/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-identity-review
description: >-
  Review how an agent authenticates and how a user's authority is delegated to
  it, including token exchange, on-behalf-of chains, service accounts and
  non-human identity. Use when asked who an agent is calling as, whether a
  delegation is auditable, why a downstream system sees the wrong principal, or
  how to scope an agent's credentials.
allowed-tools: Read, Grep, Glob
---

# Agent identity and delegation review

Three identities are in play whenever an agent acts, and most incidents come
from collapsing them:

- the **user** whose request started the work
- the **agent** doing the work — a workload identity, not a person
- the **actor chain** connecting them, which is what an auditor needs

A service account shared by every agent answers "what ran" and destroys "for
whom". That is the gap non-human identity exists to close.

## When to use this

Reviewing an agent's auth design, an OBO/token-exchange implementation, a
gateway that fronts an agent, or any log where you cannot tell which user
caused an action.

## Procedure

**1 — Name the three identities.** For the flow under review, write down the
user principal, the workload identity, and where each is asserted. If the
workload identity is a long-lived shared secret, that is finding one.

**2 — Check the delegation narrows.** In RFC 8693 token exchange the issued
token must satisfy **both** rules:

- **subset of presented** — never more scope than the incoming token carried
- **within the actor's ceiling** — never more than the agent is itself allowed

Either rule alone is insufficient. Subset-only lets a highly privileged user
hand an agent authority the agent should never hold; ceiling-only lets an agent
exceed the user who asked. Test both directions explicitly.

**3 — Check the chain is preserved and not duplicated.** The `act` chain should
read `user → agent`, once. A chain that repeats the principal
(`alice → alice → agent`) usually means the head was appended twice, and it
breaks any audit query that counts hops.

**4 — Find where OBO stops.** Some downstream systems cannot consume a
delegated token — legacy databases, vendor APIs, anything with a static
credential. Identify each, and require a **choke point**: a gateway that holds
the credential, enforces per-user authorisation *before* the call, and logs the
original principal. The credential must not be reachable by the agent directly.

**5 — Check expiry and revocation.** How long is the delegated token valid, and
what stops it being replayed after the user's session ends? Just-in-time
authority that outlives the task is standing authority with extra steps.

**6 — Check the log answers the audit question.** Pick a real question — "which
user caused this row to be deleted?" — and try to answer it from the logs alone.
If you cannot, the delegation is not auditable regardless of how it is built.

## Output contract

```json
{
  "identities": {"user": "str", "workload": "str", "assertion": "str"},
  "delegation": {"mechanism": "obo|impersonation|shared_service_account|none",
                 "subset_of_presented": true, "within_actor_ceiling": true,
                 "chain": ["principal", "..."], "chain_wellformed": true,
                 "ttl_seconds": 0, "revocable": true},
  "chokepoints": [{"downstream": "str", "reason": "str",
                   "enforced_at": "gateway|service|none",
                   "credential_reachable_by_agent": false}],
  "audit": {"question": "str", "answerable_from_logs": true},
  "findings": [{"issue": "str", "severity": "critical|high|medium|low", "fix": "str"}]
}
```

## Failure modes

- **Checking only one narrowing rule.** Both, every time.
- **Accepting a shared service account because it is "internal".** Internal is
  a network property; it says nothing about attribution.
- **Treating the gateway as optional** where the downstream cannot do OBO. It
  is the only place authorisation can happen.
- **Measuring delegation by whether the call succeeded.** A call that succeeds
  with too much scope is the failure being looked for.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/identity/agent-identity-review/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

The skill loads and reports its own shape: a routing description an agent reads to decide whether this review applies, the tools it is allowed to use, and a procedure long enough to separate user, workload and agent instance and to check both narrowing rules — scope is a subset of what was presented, and within the actor's own ceiling.

## Your turn

For one agent, write down its three identities. If the workload and the user are the same value, you have inherited credentials; if the instance does not exist, you cannot revoke one run.

---

**Next → [A2.2 · Bootstrapping the first credential](https://spbreed.github.io/cyber-commons/lessons/A2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*